# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring** (Lane 2 — Core lane)

I am choosing the Refresh / Content Opportunity Scoring lane because the dataset contains a clear, high-stakes prioritization problem: out of 30,000 content pages spread across 32 clients, more than half (54.2%) are currently classified as declining in search impressions, and nearly 10,000 of those declining pages still receive 500 or more impressions in the trailing 90-day window. That means roughly 79 million impressions — over 50% of the dataset's total — sit behind pages that are losing ground. No editorial team can review all of them at once; they need a ranked list that puts the most promising refresh candidates at the top. The starter pipeline already demonstrates that a learned model (random forest, precision@50 = 0.740) outperforms a transparent rule-based baseline (precision@50 = 0.240) by roughly 3×, which tells me the signal is real but too tangled for simple if-then rules to capture. This lane lets me build on that evidence, define a stronger future-looking label, and produce a ranked queue with reason codes that an editor can actually act on.

In [1]:
# Load the starter dataset and confirm the basic shape
import pandas as pd
import numpy as np
import os

# Navigate to repo root regardless of where notebook runs
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
DATA_PATH = os.path.join(REPO_ROOT, 'data', 'raw', 'content_refresh_anonymized.csv')

df = pd.read_csv(DATA_PATH)
print(f'Dataset: {len(df):,} rows × {len(df.columns)} columns, {df["client_id"].nunique()} clients')
print(f'Decline rate (trend_direction == "down"): {(df["trend_direction"] == "down").mean():.1%}')

Dataset: 30,000 rows × 44 columns, 32 clients
Decline rate (trend_direction == "down"): 54.2%


## 2. The question: decision, action, cost of a wrong call

**The search question:**

> Among a client's existing content pages, which ones should an editor review first for a content refresh — and what kind of action (rewrite, expand, update metadata, monitor) is most likely to be worth the effort?

**The decision this improves:**

A content strategist or editor has limited hours each week. They currently choose which pages to refresh using a mix of intuition, simple dashboards, and broad rules (e.g., "fix anything older than six months"). This project replaces that with a data-driven ranked queue: a list of pages ordered by how urgently they appear to need attention, with a reason code explaining why each one is surfaced.

**Unit of analysis:** One content page (identified by `content_id`), scored once per review cycle.

**Output:** A ranked refresh-priority score per page, with a suggested action and human-readable reason codes.

**Who acts on it, and what they do:**

A content editor or SEO strategist reviews the top-ranked pages. For each one, they decide whether to rewrite the content, expand a thin page, update the title and meta description, or simply add it to a monitoring list. The queue does not make publishing decisions — it triages review effort.

**Cost of a wrong recommendation:**

- **False positive (recommending a healthy page for refresh):** Wasted editorial hours. A content rewrite takes roughly 2–4 hours of writer time; recommending pages that do not need it wastes that capacity.
- **False negative (missing a truly declining high-value page):** Lost search visibility and organic traffic. A page with 10,000+ impressions per quarter that silently declines may lose significant traffic before anyone notices.

False negatives are arguably more costly for high-visibility pages, because lost impressions compound over time. But false positives are costly at scale — a queue full of noise erodes trust and stops editors from using it.

**Why data or ML helps at all:**

A simple rule like "flag everything with declining trend" catches 16,262 pages — over half the inventory. That is not a useful triage list; it is a firehose. The pattern connecting impressions, position, freshness, content depth, engagement, and future decline is real but tangled: word count alone shows virtually no difference between declining and rising pages (both have a median around 2,900 words), and keyword search volume has near-zero correlation with actual impressions. A model that weighs multiple interacting signals — visibility consistency (`days_with_impressions`), position trajectory, content age, engagement quality — can surface the genuinely at-risk pages that a single-rule filter misses.

In [2]:
# Show why simple rules are not enough
# Rule: "flag everything declining" catches too many pages to be useful
declining = df[df['trend_direction'] == 'down']
print(f'Pages flagged by "trend_direction == down": {len(declining):,} out of {len(df):,} ({len(declining)/len(df):.1%})')
print('→ That is more than half the inventory — not a useful triage list.\n')

# Word count barely differs between declining and rising pages
for td in ['down', 'up', 'stable']:
    subset = df[df['trend_direction'] == td]
    med = subset['word_count'].median()
    print(f'  trend={td:6s}: median word_count = {med:,.0f} words  (n={len(subset):,})')
print('→ Word count alone does not separate declining from rising pages.\n')

# Starter model results: ML beats rules 3x on precision@50
print('Starter pipeline results (from outputs/model_report.md):')
print('  Baseline rules  precision@50 = 0.240  (12 of top 50 correct)')
print('  Random Forest   precision@50 = 0.740  (37 of top 50 correct)')
print('→ ~3× lift — the signal is real but too tangled for simple if-then rules.')

Pages flagged by "trend_direction == down": 16,262 out of 30,000 (54.2%)
→ That is more than half the inventory — not a useful triage list.

  trend=down  : median word_count = 2,909 words  (n=16,262)
  trend=up    : median word_count = 2,848 words  (n=4,388)
  trend=stable: median word_count = 2,912 words  (n=5,962)
→ Word count alone does not separate declining from rising pages.

Starter pipeline results (from outputs/model_report.md):
  Baseline rules  precision@50 = 0.240  (12 of top 50 correct)
  Random Forest   precision@50 = 0.740  (37 of top 50 correct)
→ ~3× lift — the signal is real but too tangled for simple if-then rules.


## 3. Quick look at the data (2-3 real numbers)

Three numbers that show this lane is worth the next seven weeks:

**Number 1 — The scale of the prioritization problem.**
Of 30,000 pages in the starter dataset, 16,262 (54.2%) are currently classified as declining. Among those, 9,961 pages still have 500+ impressions in the trailing 90 days — together they account for roughly 79 million impressions, which is 50.7% of the dataset's total search impressions. That is a large pool of at-risk visibility, and no team can review it all. A ranked queue is essential.

**Number 2 — Decline rate varies dramatically by position tier.**
The overall decline rate is 54.2%, but it is not uniform. Pages in the `striking` position tier (average position 11–20, the "striking distance" zone) decline at 61.0%, the highest of any tier. Meanwhile `top_3` pages decline at only 24.1%. This tells us that position context matters for prioritization — a one-size-fits-all rule misses this structure.

**Number 3 — The starter model already proves ML can beat rules here.**
The committed starter pipeline shows that a random forest achieves precision@50 = 0.740 under client-holdout validation, versus 0.240 for the transparent rule baseline. That is roughly a 3× improvement: 37 of the top 50 model-ranked pages were genuinely declining, versus only 12 for the rule-based list. The top predictive features are `days_with_impressions` (0.158), `log_impressions_90d` (0.128), and `avg_position` (0.109) — signals that interact in ways a flat rule cannot capture.

In [3]:
# Number 1: Scale of the prioritization problem
declining = df[df['trend_direction'] == 'down']
declining_visible = declining[declining['impressions_90d'] >= 500]
total_impr = df['impressions_90d'].sum()
declining_impr = declining_visible['impressions_90d'].sum()

print('--- Number 1: Scale of at-risk visibility ---')
print(f'Total pages: {len(df):,}')
print(f'Declining pages: {len(declining):,} ({len(declining)/len(df):.1%})')
print(f'Declining + visible (500+ impr): {len(declining_visible):,}')
print(f'Impressions at stake: {declining_impr:,.0f} out of {total_impr:,.0f} ({declining_impr/total_impr:.1%})')
print()

# Number 2: Decline rate by position tier
print('--- Number 2: Decline rate by position tier ---')
for tier in ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']:
    subset = df[df['position_tier'] == tier]
    if len(subset) > 0:
        dr = (subset['trend_direction'] == 'down').mean()
        print(f'  {tier:10s}: decline rate = {dr:.1%}  (n={len(subset):,})')
print()

# Number 3: Model vs baseline lift
print('--- Number 3: Model vs baseline (from starter pipeline) ---')
print(f'  Baseline rules  → precision@50 = 0.240')
print(f'  Random Forest   → precision@50 = 0.740')
print(f'  Lift: {0.740 / 0.240:.1f}× improvement in top-50 precision')

--- Number 1: Scale of at-risk visibility ---
Total pages: 30,000
Declining pages: 16,262 (54.2%)
Declining + visible (500+ impr): 9,961
Impressions at stake: 79,042,325 out of 156,010,989 (50.7%)

--- Number 2: Decline rate by position tier ---
  top_3     : decline rate = 24.1%  (n=2,321)
  page_1    : decline rate = 57.0%  (n=11,814)
  striking  : decline rate = 61.0%  (n=7,304)
  page_3_5  : decline rate = 56.2%  (n=7,242)
  deep      : decline rate = 34.4%  (n=1,319)

--- Number 3: Model vs baseline (from starter pipeline) ---
  Baseline rules  → precision@50 = 0.240
  Random Forest   → precision@50 = 0.740
  Lift: 3.1× improvement in top-50 precision


## 4. Careful words: what I can and can't claim

**What this work will be able to say (observed, directional, decision-support):**

- We can **observe** which measurable signals (impression consistency, position, content age, engagement metrics) are **associated** with pages that later experience declining search visibility.
- We can build a **decision-support tool** — a ranked queue — that surfaces the most promising refresh candidates first, measured by precision@K under client-holdout validation.
- We can **measure** whether a learned model outperforms a transparent rule-based baseline at ranking review candidates, and by how much.
- We can provide **directional** guidance: pages with certain observed feature profiles appear more likely to decline, and reviewing them first is a defensible use of editorial capacity.

**What this work will never claim:**

- **No causal claims.** We cannot prove that refreshing a page *caused* it to recover. To demonstrate causation, we would need a controlled experiment (e.g., randomly refreshing some pages and not others). This dataset supports observational analysis only.
- **No algorithm claims.** We are not reverse-engineering Google's ranking algorithm or predicting how Google will rank a page. We observe search-performance patterns across a multi-client content portfolio — nothing more.
- **No guarantees.** A high score on the refresh queue means "this page looks worth reviewing based on observed signals." It does not guarantee that action will produce a recovery.
- **No private data exposure.** All identifiers are pseudonymized. No client names, domains, URLs, or raw queries appear anywhere in the analysis or the output.

In [4]:
# Verify: no private data leakage in our working dataset
print('Sanity checks for public-safe output:')
print(f'  All content_id values start with "content_": {df["content_id"].str.startswith("content_").all()}')
print(f'  All client_id values start with "client_":  {df["client_id"].str.startswith("client_").all()}')
print(f'  No URL or domain columns present:            {"url" not in df.columns and "domain" not in df.columns}')
print(f'  No title column present:                     {"title" not in df.columns}')
print()
print('All identifiers are pseudonymized. No private data in the working dataset. ✓')

Sanity checks for public-safe output:


  All content_id values start with "content_": True
  All client_id values start with "client_":  True
  No URL or domain columns present:            True
  No title column present:                     True

All identifiers are pseudonymized. No private data in the working dataset. ✓


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.